In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/MoleOOD

/content/drive/MyDrive/MoleOOD


In [ ]:
!git clone https://github.com/yangnianzu0515/MoleOOD.git

fatal: destination path 'MoleOOD' already exists and is not an empty directory.


In [5]:
%cd /content/drive/MyDrive/MoleOOD/MoleOOD/DrugOOD

/content/drive/MyDrive/MoleOOD/MoleOOD/DrugOOD


In [2]:
# @title 1. 安装核心依赖 (PyG 生态)
import torch

# 1. 安装 RDKit (用于化学分子处理，被 PyG 的 from_smiles 底层调用)
!pip install rdkit

# 2. 安装 PyTorch Geometric (PyG) 的核心库
# 获取当前 Colab 的 PyTorch 版本 (例如 2.5.1+cu121)
torch_version = torch.__version__.split('+')[0]

# 安装与当前 PyTorch 版本严格对齐的底层 C++ 扩展包
# 这四个包是 PyG 能够高效运行图计算的基石
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch_version}.html

# 安装 PyG 主库
!pip install torch-geometric

# 3. 安装可能需要的辅助库 (比如 tqdm 用于进度条)
!pip install tqdm

print("✅ 所有依赖安装完毕！环境极其纯净！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 61.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.5 MB/s eta 0:00:00
✅ 所有依赖安装完毕！环境极其纯净！


In [3]:
# @title 终极 PyG 安装方案 (GPU 版本)
import torch

# 1. 获取当前环境的 PyTorch 和 CUDA 版本信息
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda
if CUDA is None:
    CUDA = 'cpu' # 兜底，万一没分配到 GPU
else:
    CUDA = "cu" + CUDA.replace('.', '')

print(f"当前 PyTorch 版本: {TORCH}")
print(f"当前 CUDA 版本 (DGL/PyG 格式): {CUDA}")

# 2. 从 PyG 官方提供的、与版本严格匹配的 URL 安装
# `-f https://...` 参数就是告诉 pip：“别去别的地方找了，就在这个网址里下载！”
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

# 3. 安装主库
!pip install torch-geometric

当前 PyTorch 版本: 2.10.0
当前 CUDA 版本 (DGL/PyG 格式): cu128
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 122.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 128.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 113.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.7 MB/s eta 0:00:00


In [9]:
!python main.py \
  --train_path data/ic50/lbap_core_ic50_scaffold_brics.json \
  --val_path data/ic50/lbap_core_ic50_scaffold_brics.json \
  --test_path data/ic50/lbap_core_ic50_scaffold_brics.json \
  --batch_size 128 \
  --epoch_ast 20 \
  --epoch_main 50 \
  --device 0

Namespace(train_path='data/ic50/lbap_core_ic50_scaffold_brics.json', val_path='data/ic50/lbap_core_ic50_scaffold_brics.json', test_path='data/ic50/lbap_core_ic50_scaffold_brics.json', emb_dim=128, num_class=2, dropout=0.1, batch_size=128, lr=0.001, device=0, seed=2022, num_domain=20, epoch_main=50, epoch_ast=20, lambda_loss=1.0, dist='uniform')
[INFO] Building Datasets...
[Dataset] 正在尝试加载大型 JSON 文件: data/ic50/lbap_core_ic50_scaffold_brics.json
[Dataset] 成功加载 train 集，共 21519 个分子。
[Dataset] 正在尝试加载大型 JSON 文件: data/ic50/lbap_core_ic50_scaffold_brics.json
[Dataset] 成功加载 ood_val 集，共 19041 个分子。
[Dataset] 正在尝试加载大型 JSON 文件: data/ic50/lbap_core_ic50_scaffold_brics.json
[Dataset] 成功加载 ood_test 集，共 19048 个分子。
[INFO] Building Models...

[INFO] Training Assistant Models - Epoch 0
100% 169/169 [02:41<00:00,  1.05it/s]
[INFO] Eq: 0.2019, ELBO: 0.2226

[INFO] Training Assistant Models - Epoch 1
100% 169/169 [02:41<00:00,  1.05it/s]
[INFO] Eq: 0.1733, ELBO: 0.1750

[INFO] Training Assistant Models - Epo

In [12]:
!python main.py \
  --train_path data/ec50/lbap_core_ec50_scaffold_brics.json \
  --val_path data/ec50/lbap_core_ec50_scaffold_brics.json \
  --test_path data/ec50/lbap_core_ec50_scaffold_brics.json \
  --batch_size 128 \
  --epoch_ast 20 \
  --epoch_main 50 \
  --device 0

Namespace(train_path='data/ec50/lbap_core_ec50_scaffold_brics.json', val_path='data/ec50/lbap_core_ec50_scaffold_brics.json', test_path='data/ec50/lbap_core_ec50_scaffold_brics.json', emb_dim=128, num_class=2, dropout=0.1, batch_size=128, lr=0.001, device=0, seed=2022, num_domain=20, epoch_main=50, epoch_ast=20, lambda_loss=1.0, dist='uniform')
[INFO] Building Datasets...
[Dataset] 正在尝试加载大型 JSON 文件: data/ec50/lbap_core_ec50_scaffold_brics.json
[Dataset] 成功加载 train 集，共 2570 个分子。
[Dataset] 正在尝试加载大型 JSON 文件: data/ec50/lbap_core_ec50_scaffold_brics.json
[Dataset] 成功加载 ood_val 集，共 2532 个分子。
[Dataset] 正在尝试加载大型 JSON 文件: data/ec50/lbap_core_ec50_scaffold_brics.json
[Dataset] 成功加载 ood_test 集，共 2533 个分子。
[INFO] Building Models...

[INFO] Training Assistant Models - Epoch 0
100% 21/21 [00:23<00:00,  1.11s/it]
[INFO] Eq: 0.3252, ELBO: 0.4632

[INFO] Training Assistant Models - Epoch 1
100% 21/21 [00:21<00:00,  1.03s/it]
[INFO] Eq: 0.1899, ELBO: 0.2053

[INFO] Training Assistant Models - Epoch 2
10

In [14]:
!git config --global user.email "2901999089@qq.com"
!git config --global user.name "sadddddtt"

# 检查一下是否配置成功
!git config --list

filter.lfs.clean=git-lfs clean -- %f
filter.lfs.smudge=git-lfs smudge -- %f
filter.lfs.process=git-lfs filter-process
filter.lfs.required=true
user.email=2901999089@qq.com
user.name=sadddddtt
core.repositoryformatversion=0
core.filemode=true
core.bare=false
core.logallrefupdates=true
remote.origin.url=https://github.com/qwqwqwqwqwqwqwqwqe/MoleOOD.git
remote.origin.fetch=+refs/heads/*:refs/remotes/origin/*


In [15]:
# 1. 初始化本地 Git 仓库
# !git init

# 2. (极其重要！) 忽略数据文件和日志文件
# 创建一个 .gitignore 文件，防止把几百MB的数据或日志推上去
!echo "data/" > .gitignore

!echo "__pycache__/" >> .gitignore
!echo "saved_models/" >> .gitignore

# 3. 将所有代码文件添加到暂存区
!git add .

# 4. 提交代码并写上提交信息
!git commit -m "Initial commit: Pure PyG reimplementation of MoleOOD"

# 5. 将本地仓库与你在 GitHub 上新建的远程仓库关联
# 注意：把下面的 URL 换成你刚才创建的仓库地址！！！
!git remote add origin https://github.com/qwqwqwqwqwqwqwqwqe/MoleOOD.git

# 6. 更改主分支名称为 main (现在的标准做法)
!git branch -M main



[main (root-commit) efac176] Initial commit: Pure PyG reimplementation of MoleOOD
 35 files changed, 1890 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 ChemistryProcess.py
 create mode 100644 GetSubStruct.py
 create mode 100644 PreProcess.py
 create mode 100644 configs/GIN_0.1_mean.py
 create mode 100644 configs/GIN_0.3_mean.py
 create mode 100644 configs/GIN_0.5_mean.py
 create mode 100644 configs/data_assay_ec50.py
 create mode 100644 configs/data_assay_ec50_recap.py
 create mode 100644 configs/data_assay_ic50.py
 create mode 100644 configs/data_assay_ic50_recap.py
 create mode 100644 configs/data_scaffold_ec50.py
 create mode 100644 configs/data_scaffold_ec50_recap.py
 create mode 100644 configs/data_scaffold_ic50.py
 create mode 100644 configs/data_scaffold_ic50_recap.py
 create mode 100644 configs/data_size_ec50.py
 create mode 100644 configs/data_size_ec50_recap.py
 create mode 100644 configs/data_size_ic50.py
 create mode 100644 configs/data_size_ic50_recap.py

In [16]:
TOKEN = "ghp_" # <--- 替换成你的真实 Token
USERNAME = "qwqwqwqwqwqwqwqwqe"
REPO_NAME = "MoleOOD"

# 构建带权限认证的远程仓库 URL
# 格式: https://<token>@github.com/<username>/<repo>.git
REMOTE_URL = f"https://{TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git"

# 1. 如果之前加过远程地址，先删掉旧的 (防止冲突)
!git remote remove origin

# 2. 添加我们刚刚构建的带 Token 的新地址
!git remote add origin {REMOTE_URL}

# 3. 发起最后的推送！(把本地的 main 分支推送到 origin 远程仓库)
!git push -u origin main

Enumerating objects: 45, done.
Counting objects: 100% (45/45), done.
Delta compression using up to 2 threads
Compressing objects: 100% (40/40), done.
Writing objects: 100% (45/45), 35.10 MiB | 2.08 MiB/s, done.
Total 45 (delta 13), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (13/13), done.
To https://github.com/qwqwqwqwqwqwqwqwqe/MoleOOD.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
